#Fase 1: Selección de un LLM cuantizado (GGUF en CPU)

**Objetivo de esta fase:** comparar varios LLM cuantizados en formato **GGUF** ejecutados en **CPU** para elegir el modelo que se usará en las fases posteriores.

**Qué se compara:**
- **LLaMA 3.1 Instruct (8B)** (mayor capacidad).
- **LLaMA 3.2 Instruct (3B)** (alternativa compacta).
- **Gemma Instruct (~4B en este GGUF)** (familia distinta, tamaño similar al 3B–4B).

**Criterios evaluados en este notebook:**
1) **Coste/recursos**: uso de RAM y benchmark de inferencia con `llama-bench`.  
2) **Calidad del texto generado** sobre un conjunto fijo de prompts:
   - Diversidad léxica (Distinct-1 / Distinct-2).
   - Diversidad de contenido (TF-IDF + similitud coseno / near-duplicates).
   - Cumplimiento de requisitos (inspirado en FollowBench con reglas).

**Salidas (archivos) generadas:**
- `generated_reviews.csv` / `generated_reviews.xlsx`: dataset base con las reseñas generadas.
- `generated_reviews_scored.csv` / `generated_reviews_scored.xlsx`: dataset enriquecido con columnas de evaluación inspirada en FollowBench.
- `distinct_results.csv` / `distinct_results.xlsx`: Distinct-1 y Distinct-2 agregados por modelo.
- `neardup_results.csv` / `neardup_results.xlsx`: métricas TF-IDF (avg_max_similarity, near_duplicate_rate) agregadas por modelo.
- `followbench_results.csv` / `followbench_results.xlsx`: métricas inspiradas en FollowBench agregadas por modelo.

Además, se muestran en pantalla tablas de resumen (Distinct, near-duplicates y FollowBench).

## Paso 1: Instalación de librerías

En esta fase se usa:
- `llama-cpp-python` para cargar y ejecutar modelos **GGUF** localmente.
- `nltk`, `pandas` y `tqdm` para generación de datasets y métricas.

In [ ]:
# Instala llama-cpp-python (wrapper Python de llama.cpp) para ejecutar modelos GGUF.
# Se usa para cargar los modelos cuantizados y generar texto con CPU.
!pip install -U llama-cpp-python

In [ ]:
# Librerías auxiliares para:
  # - manipulación de datos (pandas)
  # - progreso visual durante generación (tqdm)
  # - tokenización y n-gramas para Distinct-n (nltk)
!pip install nltk pandas tqdm

## Paso 2: Descarga y carga de modelos (GGUF)

Para cada modelo:
1) Se mide la **RAM antes de cargarlo**.
2) Se carga el modelo cuantizado desde HuggingFace con `Llama.from_pretrained()`.
3) Se mide la **RAM tras cargarlo**.
4) Se realiza una **generación corta** (test de prueba).
5) Se mide la **RAM tras generar texto**.

### Llama 3.1 Instruct (8B)

Fuente de la cuantización GGUF: https://huggingface.co/MaziyarPanahi/Meta-Llama-3.1-8B-Instruct-GGUF

In [ ]:
import psutil, os

# Medición de memoria RAM del proceso actual (antes de cargar el modelo).
process = psutil.Process(os.getpid())
print(f"RAM antes de cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
from llama_cpp import Llama

# Carga del modelo GGUF seleccionado (cuantización Q4_K_M).
# repo_id: repositorio HuggingFace
# filename: fichero GGUF concreto dentro del repo
llm_llama31 = Llama.from_pretrained(
	repo_id="MaziyarPanahi/Meta-Llama-3.1-8B-Instruct-GGUF",
	filename="Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf",
)

In [ ]:
# RAM después de cargar el modelo en memoria.
process = psutil.Process(os.getpid())
print(f"RAM tras cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
# Generación rápida de prueba para verificar que el modelo responde.
# (No es parte del dataset final; es un test rápido de ejecución)
llm_llama31.create_chat_completion(
	messages = [
		{
			"role": "user",
			"content": "Redacta un descripción positiva de 20 palabras de una cafetera italiana."
		}
	]
)

In [ ]:
# RAM tras una llamada de inferencia (puede variar).
process = psutil.Process(os.getpid())
print(f"RAM tras generar texto: {process.memory_info().rss / 1024**3:.2f} GB")

### Llama 3.2 Instruct (3B)

Fuente de la cuantización GGUF: https://huggingface.co/bartowski/Llama-3.2-3B-Instruct-GGUF

En este notebook se prueban varias cuantizaciones del mismo modelo (Q4_K_M, Q4_0 y Q5_K_S) para observar su impacto en recursos/rendimiento.

#### Cuantización: Q4_K_M

In [ ]:
import psutil, os

# RAM antes de cargar el modelo (LLaMA 3.2 Q4_K_M).
process = psutil.Process(os.getpid())
print(f"RAM antes de cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
from llama_cpp import Llama

# Carga del modelo LLaMA 3.2 3B con cuantización Q4_K_M.
llm_llama32 = Llama.from_pretrained(
	repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
	filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf",
)

In [ ]:
# RAM tras cargar el modelo.
process = psutil.Process(os.getpid())
print(f"RAM tras cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
# Tést rápido de generación (misma instrucción para comparar comportamiento).
llm_llama32.create_chat_completion(
	messages = [
		{
			"role": "user",
			"content": "Redacta un descripción positiva de 20 palabras de una cafetera italiana."
		}
	]
)

In [ ]:
# RAM tras generar texto.
process = psutil.Process(os.getpid())
print(f"RAM tras generar texto: {process.memory_info().rss / 1024**3:.2f} GB")

#### Cuantización: Q4_0

In [ ]:
import psutil, os

# RAM antes de cargar la cuantización Q4_0.
process = psutil.Process(os.getpid())
print(f"RAM antes de cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
from llama_cpp import Llama

# Carga del mismo modelo (LLaMA 3.2 3B) pero con cuantización Q4_0.
llm = Llama.from_pretrained(
	repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
	filename="Llama-3.2-3B-Instruct-Q4_0.gguf",
)


In [ ]:
# RAM tras cargar la cuantización Q4_0.
process = psutil.Process(os.getpid())
print(f"RAM tras cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
# Tést rápido de generación
llm.create_chat_completion(
	messages = [
		{
			"role": "user",
			"content": "Redacta un descripción positiva de 20 palabras de una cafetera italiana."
		}
	]
)

In [ ]:
# RAM tras generar texto con Q4_0.
process = psutil.Process(os.getpid())
print(f"RAM tras generar texto: {process.memory_info().rss / 1024**3:.2f} GB")

#### Cuantización: Q5_K_S

In [ ]:
import psutil, os

# RAM antes de cargar la cuantización Q5_K_S.
process = psutil.Process(os.getpid())
print(f"RAM antes de cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
from llama_cpp import Llama

# Carga del mismo modelo con cuantización Q5_K_S (más bits -> suele aumentar tamaño/coste).
llm = Llama.from_pretrained(
	repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
	filename="Llama-3.2-3B-Instruct-Q5_K_S.gguf",
)

In [ ]:
# RAM tras cargar Q5_K_S.
process = psutil.Process(os.getpid())
print(f"RAM tras cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
# Tést rápido de generación
llm.create_chat_completion(
	messages = [
		{
			"role": "user",
			"content": "Redacta un descripción positiva de 20 palabras de una cafetera italiana."
		}
	]
)

In [ ]:
# RAM tras generar texto con Q5_K_S.
process = psutil.Process(os.getpid())
print(f"RAM tras generar texto: {process.memory_info().rss / 1024**3:.2f} GB")

### Gemma 3 Instruct (3B)

Fuente de cuantización GGUF: https://huggingface.co/MaziyarPanahi/gemma-3-4b-it-GGUF

In [ ]:
import psutil, os

# RAM antes de cargar Gemma.
process = psutil.Process(os.getpid())
print(f"RAM antes de cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
from llama_cpp import Llama

# Carga de Gemma cuantizado (Q4_K_M).
llm_gemma = Llama.from_pretrained(
	repo_id="MaziyarPanahi/gemma-3-4b-it-GGUF",
	filename="gemma-3-4b-it.Q4_K_M.gguf",
)

In [ ]:
# RAM tras cargar Gemma.
process = psutil.Process(os.getpid())
print(f"RAM tras cargar el modelo: {process.memory_info().rss / 1024**3:.2f} GB")

In [ ]:
# Tést rápido de generación
llm_gemma.create_chat_completion(
	messages = [
		{
			"role": "user",
			"content": "Redacta un descripción positiva de 20 palabras de una cafetera italiana."
		}
	]
)

In [ ]:
# RAM tras generar texto.
process = psutil.Process(os.getpid())
print(f"RAM tras generar texto: {process.memory_info().rss / 1024**3:.2f} GB")

## Paso 3: Evaluación y comparación de los modelos

A partir de aquí se comparan modelos en dos dimensiones:

A) **Rendimiento de inferencia** (tiempo/velocidad) con `llama-bench` (herramienta de `llama.cpp`).  
B) **Calidad del texto generado**, generando un dataset controlado y calculando métricas de:
- diversidad léxica (Distinct-n),
- diversidad de contenido (TF-IDF + similitud),
- cumplimiento de requisitos (inspirado en FollowBench por reglas).

### Modelos (para tenerlos accesibles)

#### Carga

En este bloque se vuelven a cargar los modelos en variables con nombres consistentes para usarlos en las evaluaciones posteriores.

In [ ]:
from llama_cpp import Llama

# Carga LLaMA 3.1 (Q4_K_M)
llm_llama31 = Llama.from_pretrained(
	repo_id="MaziyarPanahi/Meta-Llama-3.1-8B-Instruct-GGUF",
	filename="Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf",
)

# Carga LLaMA 3.2 (Q4_K_M)
llm_llama32 = Llama.from_pretrained(
	repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
	filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf",
)

# Carga Gemma (Q4_K_M)
llm_gemma = Llama.from_pretrained(
	repo_id="MaziyarPanahi/gemma-3-4b-it-GGUF",
	filename="gemma-3-4b-it.Q4_K_M.gguf",
)


#### Ruta (copia desde caché)


Estas celdas localizan los modelos descargados dentro de la caché y los copian a `/content/`.
Esto permite pasar rutas directas a `llama-bench` con `-m /content/<modelo>.gguf`.

Llama 3.1 Instruct (8B) Q4_K_M

In [ ]:
!find /root/.cache -name "Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf" | head -n 5

In [ ]:
!cp "/root/.cache/huggingface/hub/models--MaziyarPanahi--Meta-Llama-3.1-8B-Instruct-GGUF/snapshots/1f301d86d760b435a11a56de3863bc0121bfb98f/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf" /content/
!ls -lah /content | grep "Meta-Llama-3.1-8B"

Llama 3.2 Instruct (3B) Q4_K_M

In [ ]:
!find /root/.cache -name "Llama-3.2-3B-Instruct-Q4_K_M.gguf" | head -n 5

In [ ]:
!cp "/root/.cache/huggingface/hub/models--bartowski--Llama-3.2-3B-Instruct-GGUF/snapshots/5ab33fa94d1d04e903623ae72c95d1696f09f9e8/Llama-3.2-3B-Instruct-Q4_K_M.gguf" /content/
!ls -lah /content | grep "Llama-3.2-3B"

Q4_0

In [ ]:
!find /root/.cache -name "Llama-3.2-3B-Instruct-Q4_0.gguf" | head -n 5

In [ ]:
!cp "/root/.cache/huggingface/hub/models--bartowski--Llama-3.2-3B-Instruct-GGUF/snapshots/5ab33fa94d1d04e903623ae72c95d1696f09f9e8/Llama-3.2-3B-Instruct-Q4_0.gguf" /content/
!ls -lah /content | grep "Llama-3.2-3B"

Q5_K_S

In [ ]:
!find /root/.cache -name "Llama-3.2-3B-Instruct-Q5_K_S.gguf" | head -n 5

In [ ]:
!cp "/root/.cache/huggingface/hub/models--bartowski--Llama-3.2-3B-Instruct-GGUF/snapshots/5ab33fa94d1d04e903623ae72c95d1696f09f9e8/Llama-3.2-3B-Instruct-Q5_K_S.gguf" /content/
!ls -lah /content | grep "Llama-3.2-3B"

Gemma 3 it (4B) Q4_K_S

In [ ]:
!find /root/.cache -name "gemma-3-4b-it.Q4_K_M.gguf" | head -n 5

In [ ]:
!cp "/root/.cache/huggingface/hub/models--MaziyarPanahi--gemma-3-4b-it-GGUF/snapshots/55ebd63de390fd82075d7ec5d1cf20486292d150/gemma-3-4b-it.Q4_K_M.gguf" /content/
!ls -lah /content | grep "gemma-3-4B"

### A) Inferencia (benchmark con `llama-bench`)

Se compila `llama.cpp` para obtener `llama-bench`, y se ejecuta el benchmark para:

1) Comparar **familias/modelos distintos** con cuantización similar (Q4_K_M).  
2) Comparar **cuantizaciones** del mismo modelo (LLaMA 3.2: Q4_0 vs Q4_K_M vs Q5_K_S).

Parámetros usados en `llama-bench`:
- `-t 4`: número de hilos CPU.
- `-p 512`: longitud del prompt (tokens).
- `-n 128`: tokens a generar.
- `-r 5`: repeticiones para estabilizar la medida.

In [ ]:
# descarga de llama-bench (la herramienta de evaluación de llama.cpp)
!git clone https://github.com/ggml-org/llama.cpp.git

In [ ]:
!ls

In [ ]:
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release

In [ ]:
!cd llama.cpp && cmake --build build --config Release -j 8

In [ ]:
!find llama.cpp/build -type f -executable -name "llama-bench" | head -n 20

In [ ]:
!./llama.cpp/build/bin/llama-bench --help

In [ ]:
# Comparación de Llama 3.1, Llama 3.2 y Gemma 3
!./llama.cpp/build/bin/llama-bench \
  -m /content/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf \
  -m /content/Llama-3.2-3B-Instruct-Q4_K_M.gguf \
  -m /content/gemma-3-4b-it.Q4_K_M.gguf \
  -t 4 \
  -p 512 \
  -n 128 \
  -r 5

In [ ]:
# Comparación entre cuantizaciones de Llama 3.2
!./llama.cpp/build/bin/llama-bench \
  -m /content/Llama-3.2-3B-Instruct-Q4_0.gguf \
  -m /content/Llama-3.2-3B-Instruct-Q4_K_M.gguf \
  -m /content/Llama-3.2-3B-Instruct-Q5_K_S.gguf \
  -t 4 \
  -p 512 \
  -n 128 \
  -r 5

### B) Calidad de los modelos

En esta parte se genera un dataset controlado de reseñas para comparar modelos con el mismo tipo de tarea.

Estrategia:
- Se define un conjunto fijo de prompts (`PRODUCT_PROMPTS`) con variables: producto, perfil y rating.
- Para cada combinación se generan `K_PER_PROMPT` reseñas por modelo.
- Se guardan en un DataFrame y se exportan a CSV.

### Generación datasets para evaluar

In [ ]:
import os
import pandas as pd

OUT_DIR = "/content"

def save_table(df: pd.DataFrame, name: str) -> None:
    """
    Guarda un DataFrame como CSV y Excel en OUT_DIR con el mismo nombre base.
    Ej: name="distinct_results" -> distinct_results.csv y distinct_results.xlsx
    """
    csv_path = os.path.join(OUT_DIR, f"{name}.csv")
    xlsx_path = os.path.join(OUT_DIR, f"{name}.xlsx")
    df.to_csv(csv_path, index=False)
    df.to_excel(xlsx_path, index=False)
    print(f"Guardado: {csv_path} | {xlsx_path}")

In [ ]:
# Prompts base (mismo conjunto para todos los modelos).
# Cada elemento representa un escenario de reseña: producto, tipo de comprador y rating.

PRODUCT_PROMPTS = [
    {"product": "Auriculares inalámbricos", "persona": "usuario que hace deporte", "rating": 4},
    {"product": "Cafetera espresso", "persona": "amante del café en casa", "rating": 5},
    {"product": "Teclado mecánico", "persona": "programador", "rating": 3},
    {"product": "Mochila de viaje", "persona": "viajero de fin de semana", "rating": 4},
    {"product": "Robot aspirador", "persona": "hogar con mascota", "rating": 2},
]

def build_review_prompt(product, persona, rating):
    """
    Construye el prompt en texto a partir de las variables del escenario.
    Incluye restricciones de longitud, tono y contenido.
    """
    return f"""
Escribe una reseña realista de un producto para una tienda online.

Producto: {product}
Perfil del comprador: {persona}
Valoración: {rating} de 5

Requisitos:
- Entre 90 y 130 palabras.
- Tono natural y humano.
- Incluye 2 aspectos positivos y 1 negativo.
- No uses listas.
- No menciones que eres un modelo de lenguaje.

Reseña:
""".strip()

In [ ]:
from tqdm import tqdm
import pandas as pd

# Diccionario de modelos a comparar (mismo set de prompts para todos).
llms = {
    "LLaMA 3.1 Instruct (8B, Q4_K_M)": llm_llama31,
    "LLaMA 3.2 Instruct (3B, Q4_K_M)": llm_llama32,
    "Gemma 3 Instruct (4B, Q4_K_M)": llm_gemma,
}

rows = []
K_PER_PROMPT = 10  # nº de reseñas por prompt y modelo

def generate_review(llm, prompt, max_tokens=240, temperature=0.8, top_p=0.9):
    """
    Genera una reseña llamando a `create_chat_completion` (interfaz chat).
    Devuelve el texto generado como texto.
    """
    out = llm.create_chat_completion(
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    return out["choices"][0]["message"]["content"].strip()

# Generación del dataset:
# Por cada modelo y cada escenario (prompt), se generan K_PER_PROMPT muestras.
for model_name, llm in llms.items():
    for p in tqdm(PRODUCT_PROMPTS, desc=f"Generando con {model_name}"):
        prompt = build_review_prompt(**p)
        for i in range(K_PER_PROMPT):
            text = generate_review(llm, prompt)
            rows.append({
                "model": model_name,
                "product": p["product"],
                "persona": p.get("persona", None),
                "rating": p.get("rating", None),
                "prompt": prompt,
                "sample_id": i, # id de muestra dentro del escenario
                "text": text
            })

df_gen = pd.DataFrame(rows)

# # Vista rápida para comprobar estructura del dataset generado.
display(df_gen.head())

# Guardar dataset para evaluación posterior.
save_table(df_gen, "generated_reviews")

### Distinct-n (diversidad léxica)

Se calcula Distinct-1 y Distinct-2 por modelo:
- Distinct-n = (# n-gramas únicos) / (# n-gramas totales)
- Valores más altos indican menos repetición a nivel de n-gramas.

In [ ]:
import nltk

# Recursos de tokenización necesarios para word_tokenize en Colab.
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import word_tokenize
from nltk import ngrams
import pandas as pd
from tqdm import tqdm

In [ ]:
def distinct_n(texts, n):
    """
    Calcula Distinct-n para una lista de textos.
    - Tokeniza, construye n-gramas y devuelve proporción de n-gramas únicos.
    """
    all_ngrams = []
    for text in texts:
        tokens = word_tokenize(text.lower())
        all_ngrams.extend(list(ngrams(tokens, n)))
    return len(set(all_ngrams)) / len(all_ngrams) if all_ngrams else 0.0


In [ ]:
# Usamos df_gen (dataset ya generado en el notebook).
# Si ejecutas esta parte en una sesión distinta, puedes cargarlo así:
# df_gen = pd.read_csv("/content/generated_reviews.csv")

In [ ]:
# Cálculo de Distinct-1 y Distinct-2 agregando por modelo.
results = []
for model_name, group in df_gen.groupby("model"):
    texts = group["text"].dropna().tolist()
    results.append({
        "model": model_name,
        "distinct_1": distinct_n(texts, 1),
        "distinct_2": distinct_n(texts, 2),
    })

distinct_df = pd.DataFrame(results).sort_values("model").reset_index(drop=True)
distinct_df[["distinct_1", "distinct_2"]] = distinct_df[["distinct_1", "distinct_2"]].round(4)

In [ ]:
# Mostrar resultados
display(distinct_df)

# Guardar archivos (dataset + resultados)
save_table(distinct_df, "distinct_results")

### TF-IDF + similitud coseno (diversidad de contenido)

Se representa cada reseña con TF-IDF (uni+bi-gramas) y se calcula la matriz de similitud coseno.
Para cada reseña se toma su **máxima similitud con otra reseña** (ignorando la diagonal).

Métricas por modelo:
- `avg_max_similarity`: media de la máxima similitud por reseña (cuanto más alto, más parecido entre textos).
- `near_duplicate_rate`: proporción de reseñas cuya máxima similitud supera un umbral (por defecto 0.90).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Usamos df_gen (dataset ya generado en el notebook).
# Si ejecutas esta parte en una sesión distinta, puedes cargarlo así:
# df_gen = pd.read_csv("/content/generated_reviews.csv")

In [ ]:
def near_duplicate_metrics(texts, threshold=0.90):
    """
    Calcula métricas de similitud de contenido dentro de un conjunto de textos:
    - avg_max_sim: media de la máxima similitud coseno por texto.
    - near_dup_rate: proporción de textos cuyo 'mejor match' supera el umbral.
    """
    texts = [str(t) for t in texts]
    if len(texts) < 2:
        return np.nan, np.nan

    # TF-IDF sobre uni+bi-gramas para capturar palabras y frases cortas
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    X = vec.fit_transform(texts)

    # Matriz de similitud coseno (NxN)
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, -1)  # ignorar similitud consigo mismo

    # Para cada reseña, su mayor similitud con otra reseña
    max_sim = sim.max(axis=1)  # mejor “match” de cada reseña
    avg_max_sim = float(np.mean(max_sim))

    # Proporción de casos que superan el umbral (near-duplicates)
    near_dup_rate = float(np.mean(max_sim >= threshold))

    return avg_max_sim, near_dup_rate

In [ ]:
# Cálculo avg_max_sim y near_dup_rate agregando por modelo.
rows = []
for model, g in df_gen.groupby("model"):
    per_prompt = []
    # Se evalúa por escenario (product/persona/rating) para comparar diversidad dentro del mismo prompt.
    for _, gg in g.groupby(["product", "persona", "rating"]):
        per_prompt.append(near_duplicate_metrics(gg["text"].tolist(), threshold=0.90))

    rows.append({
        "model": model,
        "avg_max_similarity": np.mean([x[0] for x in per_prompt]),
        "near_duplicate_rate": np.mean([x[1] for x in per_prompt]),
    })

neardup_df = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)

In [ ]:
# Mostrar resultados
display(neardup_df)

# Guardar archivos (dataset + resultados)
save_table(neardup_df, "neardup_results")

### FollowBench

Se implementa un score por reglas basado en los requisitos del prompt:
1) Longitud entre 90 y 130 palabras.
2) No usar listas (heurística por patrones).
3) No mencionar IA/modelo de lenguaje (regex).
4) Contener al menos 2 aspectos positivos (lista de palabras).
5) Contener al menos 1 aspecto negativo (lista de palabras).

El score final es la media de checks cumplidos (0–1).
Además se calcula el porcentaje de reseñas con cumplimiento perfecto (score = 1.0).

In [ ]:
import re
import pandas as pd

In [ ]:
# Usamos df_gen (dataset ya generado en el notebook).
# Si ejecutas esta parte en una sesión distinta, puedes cargarlo así:
# df_gen = pd.read_csv("/content/generated_reviews.csv")

In [ ]:
# Listas de palabras positivas y negativas para evaluar la condición:
# "Presencia de al menos 2 aspectos positivos, Presencia de al menos 1 aspecto negativo"

POSITIVE_WORDS = [
    "bueno", "excelente", "cómodo", "práctico", "útil",
    "recomendable", "calidad", "satisfecho", "funciona bien"
]

NEGATIVE_WORDS = [
    "pero", "aunque", "problema", "mejoraría",
    "negativo", "inconveniente", "defecto"
]

In [ ]:
# Funciones de verificación (checks individuales)

def word_count(text):
    # Cuenta palabras separando por espacios (heurística simple).
    return len(text.split())

def check_length(text):
    # Verifica que el texto esté dentro del rango exigido por el prompt.
    wc = word_count(text)
    return 90 <= wc <= 130

def check_no_lists(text):
    # Heurística: busca patrones típicos de listas (viñetas o enumeraciones).
    return not bool(re.search(r"(\n-|•|\d+\.)", text))

def check_no_ai_mentions(text):
    # Evita menciones explícitas a IA/modelo de lenguaje.
    return not bool(re.search(
        r"(modelo de lenguaje|inteligencia artificial|IA)",
        text.lower()
    ))

def check_positive_aspects(text):
    # Cuenta cuántas palabras/expresiones positivas aparecen (>=2).
    text_l = text.lower()
    return sum(word in text_l for word in POSITIVE_WORDS) >= 2

def check_negative_aspect(text):
   # Verifica presencia de al menos una señal negativa.
    text_l = text.lower()
    return any(word in text_l for word in NEGATIVE_WORDS)

In [ ]:
# score agregado: proporción de checks cumplidos (0 a 1)

def followbench_score(text):
    checks = [
        check_length(text),
        check_no_lists(text),
        check_no_ai_mentions(text),
        check_positive_aspects(text),
        check_negative_aspect(text)
    ]
    return sum(checks) / len(checks)

In [ ]:
# Crear versión enriquecida del dataset (sin modificar df_gen)
df_scored = df_gen.copy()

# Score por reseña (fila)
df_scored["followbench_score"] = df_scored["text"].dropna().apply(followbench_score)

# Tasa de cumplimiento perfecto
df_scored["followbench_perfect"] = df_scored["followbench_score"] == 1.0

# Agregar por modelo
results = []
for model_name, group in df_scored.groupby("model"):
    scores = group["followbench_score"].dropna()
    perfect = group["followbench_perfect"].dropna()

    results.append({
        "model": model_name,
        "followbench_mean": scores.mean(),
        "followbench_std": scores.std(),
        "followbench_min": scores.min(),
        "followbench_max": scores.max(),
        "perfect_follow_rate": perfect.mean(),
    })

followbench_df = pd.DataFrame(results).sort_values("model").reset_index(drop=True)

# Redondeo
cols_round = ["followbench_mean", "followbench_std", "followbench_min", "followbench_max", "perfect_follow_rate"]
followbench_df[cols_round] = followbench_df[cols_round].round(4)

In [ ]:
# Mostrar resultados
display(followbench_df)

# Guardar resultados agregados (CSV + Excel)
save_table(followbench_df, "followbench_results")

# Guardar dataset enriquecido (CSV + Excel)
save_table(df_scored, "generated_reviews_scored")